In [1]:
!pip install -q torchmetrics torch-fidelity
!pip install -q torchmetrics torch-fidelity lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.0 MB/s eta 0:00:00


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2: V15E — FULL FINAL PAPER RUNS — 3 SEEDS (FIXED)
# ══════════════════════════════════════════════════════════════════════════════
#
# This is the full V15E training cell adapted for multi-seed final paper runs.
#
# It preserves:
#   - static hinge margin = 1.0
#   - EMA generator
#   - Gap-only adaptive N_DIS controller
#   - stochastic unbiased mode-seeking scheduler
#   - best FID / best Recall checkpoint tracking
#   - per-class metrics
#   - controller history
#   - periodic checkpoints
#   - sample generation
#
# Outputs are saved under:
#   /content/drive/MyDrive/lung_gan_checkpoints_v15E_multiseed/seed_XXXX/
#
# ══════════════════════════════════════════════════════════════════════════════

import os
import math
import copy
import random
import zipfile
import time
import gc
import warnings
from collections import deque

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils import spectral_norm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
except ImportError:
    !pip install -q torchmetrics
    from torchmetrics.image.fid import FrechetInceptionDistance

try:
    import lpips
except ImportError:
    !pip install -q lpips
    import lpips

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
SEEDS = [42, 123, 777]
EPOCHS = 200

IMG_SIZE = 128
BATCH_SIZE = 64
Z_DIM = 128
EMBED_DIM = 128
BASE_CH = 512
D_BASE_CH = 64
CH_FLOOR = 64

LR_G = 2e-4
LR_D = 2e-4
BETAS = (0.0, 0.9)
EMA_DECAY = 0.999

USE_CLAHE = True
USE_HFLIP = True

BASE_MARGIN = 1.0

# V15E controller settings
CONTROLLER_WARMUP_EPOCHS = 3

N_DIS_EXPENSIVE = 2
N_DIS_CHEAP = 1

GAP_LOW = 0.10
GAP_RELAX = 0.12
CALM_PATIENCE = 40

D_VOL_LOW = 0.002
D_VOL_HIGH = 0.020
STABILITY_WINDOW = 100

USE_MODE_SEEKING = True
MS_EFFECTIVE_TARGET = 0.06
MS_EFFECTIVE_MAX = 0.15
MS_LAMBDA_APPLIED = 0.15
MS_REFRESH_EVERY = 20
LZ_HEALTHY = 0.40
LZ_COLLAPSE = 0.25

COMPUTE_METRICS = True
METRICS_EVERY = 25
SAMPLE_EVERY = 5
CHECKPOINT_EVERY = 25
METRICS_NUM_SAMPLES = 2000
PR_K = 3
LPIPS_NUM_PAIRS = 300
COMPUTE_PER_CLASS_METRICS = True

SAVE_SAMPLES = True
SAVE_HISTORY = True

CKPT_ROOT = "/content/drive/MyDrive/lung_gan_checkpoints_v15E_multiseed"
os.makedirs(CKPT_ROOT, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ══════════════════════════════════════════════════════════════════════════════
# SEED UTIL
# ══════════════════════════════════════════════════════════════════════════════
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ══════════════════════════════════════════════════════════════════════════════
# MOUNT DRIVE & DATASET (FIXED)
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

ZIP_PATH = "/content/drive/MyDrive/Lung Segmentation Data V2.zip"
EXTRACT_ROOT = "/content/lung_data"

if not os.path.exists(EXTRACT_ROOT):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_ROOT)

data_root = None
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "Train" in dirs:
        data_root = os.path.join(root, "Train")
        break

if data_root is None:
    raise FileNotFoundError("Could not find Train folder.")

print("Data root:", data_root)

# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════
class CLAHETransform:
    def __call__(self, img):
        img_np = np.array(img)

        if len(img_np.shape) == 3:
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)

            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            l = clahe.apply(l)

            lab = cv2.merge((l, a, b))
            img_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        else:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            img_np = clahe.apply(img_np)

        return Image.fromarray(img_np)


tfm_list = [transforms.Resize((IMG_SIZE, IMG_SIZE))]

if USE_CLAHE:
    tfm_list.append(CLAHETransform())

if USE_HFLIP:
    tfm_list.append(transforms.RandomHorizontalFlip(p=0.5))

tfm_list += [
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
]

transform = transforms.Compose(tfm_list)
dataset = datasets.ImageFolder(data_root, transform=transform)

NUM_CLASSES = len(dataset.classes)
print(f"Classes ({NUM_CLASSES}): {dataset.classes}")
print(f"Dataset size: {len(dataset)}")

# ══════════════════════════════════════════════════════════════════════════════
# MODELS
# ══════════════════════════════════════════════════════════════════════════════
class ConditionalBatchNorm2d(nn.Module):
    def __init__(self, num_features, embed_dim):
        super().__init__()
        self.num_features = num_features

        self.bn = nn.BatchNorm2d(num_features, affine=False)

        self.gamma_fc = nn.Linear(embed_dim, num_features)
        self.beta_fc = nn.Linear(embed_dim, num_features)

        nn.init.normal_(self.gamma_fc.weight, 0.0, 0.02)
        nn.init.ones_(self.gamma_fc.bias)

        nn.init.normal_(self.beta_fc.weight, 0.0, 0.02)
        nn.init.zeros_(self.beta_fc.bias)

    def forward(self, x, class_embed):
        out = self.bn(x)

        gamma = self.gamma_fc(class_embed).view(-1, self.num_features, 1, 1)
        beta = self.beta_fc(class_embed).view(-1, self.num_features, 1, 1)

        return out * gamma + beta


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, embed_dim):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn = ConditionalBatchNorm2d(out_ch, embed_dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, class_embed):
        x = self.up(x)
        x = self.conv(x)
        x = self.bn(x, class_embed)
        return self.act(x)


class ConditionalGenerator(nn.Module):
    def __init__(
        self,
        z_dim,
        embed_dim,
        num_classes,
        base_ch,
        img_size,
        ch_floor=CH_FLOOR,
    ):
        super().__init__()

        self.init_size = 4
        self.base_ch = base_ch
        self.embed_dim = embed_dim

        num_stages = int(math.log2(img_size // self.init_size))
        assert self.init_size * (2 ** num_stages) == img_size

        self.label_embed = nn.Embedding(num_classes, embed_dim)
        nn.init.normal_(self.label_embed.weight, 0.0, 0.02)

        self.fc = nn.Linear(
            z_dim + embed_dim,
            base_ch * self.init_size * self.init_size,
        )

        channels = [base_ch]
        ch = base_ch

        for _ in range(num_stages):
            ch = max(ch // 2, ch_floor)
            channels.append(ch)

        self.up_blocks = nn.ModuleList()

        in_ch = base_ch
        for out_ch in channels[1:]:
            self.up_blocks.append(UpBlock(in_ch, out_ch, embed_dim))
            in_ch = out_ch

        self.to_rgb = nn.Sequential(
            nn.Conv2d(in_ch, 3, kernel_size=3, padding=1),
            nn.Tanh(),
        )

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        name = m.__class__.__name__

        if "Conv" in name or "Linear" in name:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias.data)

        elif "BatchNorm" in name:
            if getattr(m, "weight", None) is not None:
                nn.init.normal_(m.weight.data, 1.0, 0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias.data)

    def forward(self, z, labels):
        class_embed = self.label_embed(labels)

        x = torch.cat([z, class_embed], dim=1)
        x = self.fc(x).view(-1, self.base_ch, self.init_size, self.init_size)

        for block in self.up_blocks:
            x = block(x, class_embed)

        return self.to_rgb(x)


class ProjectionDiscriminator(nn.Module):
    def __init__(self, num_classes, img_size, base_ch):
        super().__init__()

        num_stages = int(math.log2(img_size // 4))

        layer_channels = []
        ch = base_ch

        for _ in range(num_stages):
            layer_channels.append(ch)
            ch = min(ch * 2, 512)

        blocks = []
        in_ch = 3

        for out_ch in layer_channels:
            conv = nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
            nn.init.normal_(conv.weight, 0.0, 0.02)

            blocks.append(spectral_norm(conv))
            blocks.append(nn.LeakyReLU(0.2, inplace=True))

            in_ch = out_ch

        self.features = nn.Sequential(*blocks)
        self.feature_dim = in_ch
        self.pool = nn.AdaptiveAvgPool2d(1)

        fc = nn.Linear(self.feature_dim, 1)
        nn.init.normal_(fc.weight, 0.0, 0.02)
        nn.init.zeros_(fc.bias)
        self.fc = spectral_norm(fc)

        embed = nn.Embedding(num_classes, self.feature_dim)
        nn.init.normal_(embed.weight, 0.0, 0.02)
        self.embed = spectral_norm(embed)

    def forward(self, x, labels):
        h = self.features(x)
        h = self.pool(h).flatten(1)

        out = self.fc(h).view(-1)
        proj = (self.embed(labels) * h).sum(dim=1)

        return out + proj


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()

        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s_p, p in zip(self.shadow.parameters(), model.parameters()):
            s_p.mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)

        for s_b, b in zip(self.shadow.buffers(), model.buffers()):
            s_b.copy_(b)

# ══════════════════════════════════════════════════════════════════════════════
# LOSSES
# ══════════════════════════════════════════════════════════════════════════════
def d_hinge_loss(real_pred, fake_pred, margin=BASE_MARGIN):
    return F.relu(margin - real_pred).mean() + F.relu(margin + fake_pred).mean()


def g_hinge_loss(fake_pred):
    return -fake_pred.mean()


def mode_seeking_loss(img1, img2, z1, z2, eps=1e-5):
    img_dist = torch.mean(torch.abs(img1 - img2))
    z_dist = torch.mean(torch.abs(z1 - z2))

    lz = img_dist / (z_dist + eps)

    return 1.0 / (lz + eps), lz

# ══════════════════════════════════════════════════════════════════════════════
# V15E CONTROLLER (FIXED state_dict)
# ══════════════════════════════════════════════════════════════════════════════
class GapOnlyAdaptiveController:
    def __init__(self, window=STABILITY_WINDOW, ema=0.98, eps=1e-8):
        self.window = window
        self.ema = ema
        self.eps = eps

        self.d_loss_window = deque(maxlen=window)

        self.n_dis = N_DIS_EXPENSIVE
        self.calm_counter = 0

        self.ema_gap = None
        self.ema_vol = 0.0
        self.ema_div = 0.0

        self.effective_ms = MS_EFFECTIVE_TARGET
        self.last_lz = LZ_HEALTHY

        self.last_gap = 0.0
        self.last_vol = 0.0
        self.last_div = 0.0

    def _vol_score(self):
        if len(self.d_loss_window) < 20:
            return 0.0

        arr = np.array(self.d_loss_window, dtype=np.float32)
        mean_val = float(arr.mean())
        std_val = float(arr.std())

        vol = std_val / (abs(mean_val) + self.eps)

        return float(np.clip(
            (vol - D_VOL_LOW) / (D_VOL_HIGH - D_VOL_LOW),
            0.0,
            1.0,
        ))

    def _div_score(self, lz):
        return float(np.clip(
            (LZ_HEALTHY - lz) / (LZ_HEALTHY - LZ_COLLAPSE),
            0.0,
            1.0,
        ))

    def update(self, d_loss, d_real, d_fake, lz=None):
        d_loss = float(d_loss)
        d_real = float(d_real)
        d_fake = float(d_fake)

        if not (np.isfinite(d_loss) and np.isfinite(d_real) and np.isfinite(d_fake)):
            return

        self.d_loss_window.append(d_loss)

        gap = d_real - d_fake

        if self.ema_gap is None:
            self.ema_gap = gap
        else:
            self.ema_gap = self.ema * self.ema_gap + (1.0 - self.ema) * gap

        vol_score = self._vol_score()
        self.ema_vol = self.ema * self.ema_vol + (1.0 - self.ema) * vol_score

        if lz is not None and np.isfinite(lz):
            self.last_lz = float(lz)
            div_score = self._div_score(self.last_lz)
            self.ema_div = self.ema * self.ema_div + (1.0 - self.ema) * div_score

        weak = self.ema_gap < GAP_LOW
        relax = self.ema_gap > GAP_RELAX

        if weak:
            self.n_dis = N_DIS_EXPENSIVE
            self.calm_counter = 0

        elif self.n_dis == N_DIS_EXPENSIVE:
            if relax:
                self.calm_counter += 1
            else:
                self.calm_counter = max(0, self.calm_counter - 1)

            if self.calm_counter >= CALM_PATIENCE:
                self.n_dis = N_DIS_CHEAP
                self.calm_counter = 0

        target_ms = MS_EFFECTIVE_TARGET + (MS_EFFECTIVE_MAX - MS_EFFECTIVE_TARGET) * self.ema_div
        self.effective_ms = self.ema * self.effective_ms + (1.0 - self.ema) * target_ms

        self.last_gap = float(self.ema_gap)
        self.last_vol = float(self.ema_vol)
        self.last_div = float(self.ema_div)

    def decide_ms(self, global_step):
        if global_step % MS_REFRESH_EVERY == 0:
            return True, MS_LAMBDA_APPLIED

        p = float(np.clip(self.effective_ms / MS_LAMBDA_APPLIED, 0.0, 1.0))

        if random.random() < p:
            return True, MS_LAMBDA_APPLIED

        return False, 0.0

    def state_dict(self):  # <--- FIXED: Added 'self' argument
        return {
            "n_dis": self.n_dis,
            "calm_counter": self.calm_counter,
            "ema_gap": self.ema_gap,
            "ema_vol": self.ema_vol,
            "ema_div": self.ema_div,
            "effective_ms": self.effective_ms,
            "last_lz": self.last_lz,
            "d_loss_window": list(self.d_loss_window),
        }

# ══════════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def _knn_manifold_precision_recall(real_feats, fake_feats, k=3):
    def knn_radii(feats, k):
        d = torch.cdist(feats, feats)
        d.fill_diagonal_(float("inf"))
        return d.topk(k, largest=False).values[:, -1]

    real_radii = knn_radii(real_feats, k)
    fake_radii = knn_radii(fake_feats, k)

    d_fake_to_real = torch.cdist(fake_feats, real_feats)
    precision = (d_fake_to_real <= real_radii.unsqueeze(0)).any(dim=1).float().mean().item()

    d_real_to_fake = torch.cdist(real_feats, fake_feats)
    recall = (d_real_to_fake <= fake_radii.unsqueeze(0)).any(dim=1).float().mean().item()

    return precision, recall


@torch.no_grad()
def _lpips_diversity(
    generator,
    lpips_metric,
    device,
    num_classes,
    z_dim,
    n_pairs=300,
    batch=32,
    fixed_class=None,
):
    dists = []
    remaining = n_pairs

    while remaining > 0:
        b = min(batch, remaining)

        if fixed_class is None:
            labels = torch.randint(0, num_classes, (b,), device=device)
        else:
            labels = torch.full((b,), fixed_class, device=device, dtype=torch.long)

        z1 = torch.randn(b, z_dim, device=device)
        z2 = torch.randn(b, z_dim, device=device)

        img1 = generator(z1, labels)
        img2 = generator(z2, labels)

        dists.append(lpips_metric(img1, img2).view(-1))

        remaining -= b

    return torch.cat(dists).mean().item()


def compute_metrics(
    generator,
    real_loader,
    device,
    fid_metric,
    lpips_metric,
    num_samples=2000,
    z_dim=128,
    num_classes=2,
    pr_k=3,
    lpips_num_pairs=300,
    fixed_class=None,
):
    generator.eval()

    proc_batch = 16

    fid_metric.reset()

    real_feats = []
    fake_feats = []

    loader_iter = iter(real_loader)

    with torch.no_grad():
        for _ in range(num_samples // proc_batch):
            try:
                real_imgs, real_labels = next(loader_iter)
            except StopIteration:
                break

            real_imgs = real_imgs[:proc_batch].to(device)
            real_labels = real_labels[:proc_batch].to(device)

            z = torch.randn(real_imgs.size(0), z_dim, device=device)
            fake_imgs = generator(z, real_labels)

            real_uint8 = ((real_imgs + 1) / 2 * 255).clamp(0, 255).to(torch.uint8)
            fake_uint8 = ((fake_imgs + 1) / 2 * 255).clamp(0, 255).to(torch.uint8)

            fid_metric.update(real_uint8, real=True)
            fid_metric.update(fake_uint8, real=False)

            real_feats.append(fid_metric.inception(real_uint8).float())
            fake_feats.append(fid_metric.inception(fake_uint8).float())

    fid_score = fid_metric.compute().item()

    real_feats = torch.cat(real_feats, dim=0)
    fake_feats = torch.cat(fake_feats, dim=0)

    precision, recall = _knn_manifold_precision_recall(
        real_feats,
        fake_feats,
        k=pr_k,
    )

    lpips_div = _lpips_diversity(
        generator,
        lpips_metric,
        device,
        num_classes,
        z_dim,
        n_pairs=lpips_num_pairs,
        fixed_class=fixed_class,
    )

    generator.train()

    return fid_score, precision, recall, lpips_div

# ══════════════════════════════════════════════════════════════════════════════
# CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════
def save_checkpoint(path, epoch_num, G, D, G_ema, opt_G, opt_D, controller, last_metrics):
    torch.save({
        "epoch": epoch_num,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "G_ema": G_ema.shadow.state_dict(),
        "opt_G": opt_G.state_dict(),
        "opt_D": opt_D.state_dict(),
        "classes": dataset.classes,
        "controller": controller.state_dict(),
        "fid": last_metrics["fid"],
        "precision": last_metrics["precision"],
        "recall": last_metrics["recall"],
        "lpips_diversity": last_metrics["lpips_diversity"],
    }, path)

# ══════════════════════════════════════════════════════════════════════════════
# MULTI-SEED TRAINING
# ══════════════════════════════════════════════════════════════════════════════
all_results = []

for seed in SEEDS:
    print("\n" + "=" * 80)
    print(f"V15E | SEED {seed}")
    print("=" * 80)

    set_seed(seed)

    seed_dir = os.path.join(CKPT_ROOT, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    g = torch.Generator()
    g.manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        drop_last=True,
        pin_memory=True,
        persistent_workers=True,
        worker_init_fn=seed_worker,
        generator=g,
    )

    class_loaders = {}
    if COMPUTE_PER_CLASS_METRICS:
        targets = np.array(dataset.targets)

        for c in range(NUM_CLASSES):
            idxs = np.where(targets == c)[0].tolist()

            class_loaders[c] = DataLoader(
                Subset(dataset, idxs),
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=2,
                drop_last=True,
                pin_memory=True,
                persistent_workers=True,
                worker_init_fn=seed_worker,
                generator=g,
            )

    G = ConditionalGenerator(
        Z_DIM,
        EMBED_DIM,
        NUM_CLASSES,
        BASE_CH,
        IMG_SIZE,
    ).to(device)

    D = ProjectionDiscriminator(
        NUM_CLASSES,
        IMG_SIZE,
        D_BASE_CH,
    ).to(device)

    G_ema = EMA(G, EMA_DECAY)

    opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=BETAS)
    opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=BETAS)

    controller = GapOnlyAdaptiveController()

    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(device)

    lpips_metric = lpips.LPIPS(net="alex").to(device)
    for p in lpips_metric.parameters():
        p.requires_grad_(False)

    fixed_z = torch.randn(NUM_CLASSES * 4, Z_DIM, device=device)
    fixed_labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(4)[:NUM_CLASSES * 4]

    metric_history = {
        "epoch": [],
        "fid": [],
        "precision": [],
        "recall": [],
        "lpips_diversity": [],
    }

    metric_history["per_class"] = {
        cname: {
            "epoch": [],
            "fid": [],
            "precision": [],
            "recall": [],
            "lpips_diversity": [],
        }
        for cname in dataset.classes
    }

    per_class_recall_leader_history = []

    loss_history = {
        "epoch": [],
        "D": [],
        "G": [],
        "MS_ratio": [],
        "D_real": [],
        "D_fake": [],
    }

    controller_history = {
        "epoch": [],
        "n_dis": [],
        "ms_frac": [],
        "eff_ms": [],
        "gap": [],
        "vol": [],
        "div": [],
    }

    best_recall = -1.0
    best_fid = float("inf")

    best_recall_info = {}
    best_fid_info = {}

    last_metrics = {
        "fid": None,
        "precision": None,
        "recall": None,
        "lpips_diversity": None,
    }

    start_time = time.time()
    global_step = 0

    for epoch in range(EPOCHS):
        if epoch == CONTROLLER_WARMUP_EPOCHS:
            controller.n_dis = N_DIS_EXPENSIVE
            controller.calm_counter = 0

        d_loss_sum = 0.0
        g_loss_sum = 0.0
        ms_lz_sum = 0.0
        real_pred_sum = 0.0
        fake_pred_sum = 0.0

        n_dis_sum = 0.0
        ms_computed_count = 0
        eff_ms_sum = 0.0

        gap_sum = 0.0
        vol_sum = 0.0
        div_sum = 0.0

        n_batches = 0

        pbar = tqdm(loader, desc=f"Seed {seed} Epoch {epoch + 1}/{EPOCHS}")

        for real_imgs, labels in pbar:
            real_imgs = real_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            bs = real_imgs.size(0)

            if epoch < CONTROLLER_WARMUP_EPOCHS:
                current_n_dis = N_DIS_EXPENSIVE
                ms_compute = True
                current_ms_lambda = MS_EFFECTIVE_TARGET
            else:
                current_n_dis = max(1, int(controller.n_dis))
                ms_compute, current_ms_lambda = controller.decide_ms(global_step)

            # Discriminator
            for _ in range(current_n_dis):
                z = torch.randn(bs, Z_DIM, device=device)

                with torch.no_grad():
                    fake_imgs = G(z, labels)

                d_real = D(real_imgs, labels)
                d_fake = D(fake_imgs, labels)

                d_loss = d_hinge_loss(d_real, d_fake, margin=BASE_MARGIN)

                opt_D.zero_grad(set_to_none=True)
                d_loss.backward()
                nn.utils.clip_grad_norm_(D.parameters(), max_norm=1000)
                opt_D.step()

            # Generator
            z = torch.randn(bs, Z_DIM, device=device)
            fake_imgs = G(z, labels)

            g_loss = g_hinge_loss(D(fake_imgs, labels))

            ms_lz_val = torch.tensor(controller.last_lz, device=device)
            lz_for_controller = None

            if USE_MODE_SEEKING and ms_compute:
                z2 = torch.randn(bs, Z_DIM, device=device)
                fake_imgs2 = G(z2, labels)

                ms_term, ms_lz_val = mode_seeking_loss(fake_imgs, fake_imgs2, z, z2)
                g_loss = g_loss + current_ms_lambda * ms_term

                lz_for_controller = ms_lz_val.item()
                ms_computed_count += 1

            opt_G.zero_grad(set_to_none=True)
            g_loss.backward()
            nn.utils.clip_grad_norm_(G.parameters(), max_norm=1000)
            opt_G.step()

            G_ema.update(G)

            controller.update(
                d_loss=d_loss.item(),
                d_real=d_real.mean().item(),
                d_fake=d_fake.mean().item(),
                lz=lz_for_controller,
            )

            global_step += 1

            d_loss_sum += d_loss.item()
            g_loss_sum += g_loss.item()
            ms_lz_sum += ms_lz_val.item()

            real_pred_sum += d_real.mean().item()
            fake_pred_sum += d_fake.mean().item()

            n_dis_sum += current_n_dis
            eff_ms_sum += controller.effective_ms

            gap_sum += controller.last_gap
            vol_sum += controller.last_vol
            div_sum += controller.last_div

            n_batches += 1

            pbar.set_postfix(
                D_loss=d_loss.item(),
                G_loss=g_loss.item(),
                ND=current_n_dis,
                MS_on=int(ms_compute),
                Gap=f"{controller.last_gap:.3f}",
            )

        avg_d = d_loss_sum / n_batches
        avg_g = g_loss_sum / n_batches
        avg_ms_ratio = ms_lz_sum / n_batches
        avg_real = real_pred_sum / n_batches
        avg_fake = fake_pred_sum / n_batches

        avg_n_dis = n_dis_sum / n_batches
        ms_computed_frac = ms_computed_count / n_batches
        avg_eff_ms = eff_ms_sum / n_batches

        avg_gap = gap_sum / n_batches
        avg_vol = vol_sum / n_batches
        avg_div = div_sum / n_batches

        loss_history["epoch"].append(epoch + 1)
        loss_history["D"].append(avg_d)
        loss_history["G"].append(avg_g)
        loss_history["MS_ratio"].append(avg_ms_ratio)
        loss_history["D_real"].append(avg_real)
        loss_history["D_fake"].append(avg_fake)

        controller_history["epoch"].append(epoch + 1)
        controller_history["n_dis"].append(avg_n_dis)
        controller_history["ms_frac"].append(ms_computed_frac)
        controller_history["eff_ms"].append(avg_eff_ms)
        controller_history["gap"].append(avg_gap)
        controller_history["vol"].append(avg_vol)
        controller_history["div"].append(avg_div)

        print(
            f"Seed {seed} Epoch {epoch + 1}/{EPOCHS} | "
            f"D={avg_d:.4f} | G={avg_g:.4f} | "
            f"D(real)={avg_real:.3f} D(fake)={avg_fake:.3f} | "
            f"ND={avg_n_dis:.2f} | MS={ms_computed_frac:.1%} | "
            f"effλ={avg_eff_ms:.3f} | Gap={avg_gap:.3f}"
        )

        # Metrics
        if COMPUTE_METRICS and ((epoch + 1) % METRICS_EVERY == 0 or epoch == 0):
            fid_val, prec_val, rec_val, lpips_val = compute_metrics(
                G_ema.shadow,
                loader,
                device,
                fid_metric,
                lpips_metric,
                num_samples=METRICS_NUM_SAMPLES,
                z_dim=Z_DIM,
                num_classes=NUM_CLASSES,
                pr_k=PR_K,
                lpips_num_pairs=LPIPS_NUM_PAIRS,
            )

            metric_history["epoch"].append(epoch + 1)
            metric_history["fid"].append(fid_val)
            metric_history["precision"].append(prec_val)
            metric_history["recall"].append(rec_val)
            metric_history["lpips_diversity"].append(lpips_val)

            print(
                f"  Metrics: FID={fid_val:.2f}  P={prec_val:.4f}  "
                f"R={rec_val:.4f}  LPIPS={lpips_val:.4f}"
            )

            last_metrics.update(
                fid=fid_val,
                precision=prec_val,
                recall=rec_val,
                lpips_diversity=lpips_val,
            )

            if rec_val > best_recall:
                best_recall = rec_val

                best_recall_info = {
                    "epoch": epoch + 1,
                    "fid": fid_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "lpips_diversity": lpips_val,
                }

                save_checkpoint(
                    os.path.join(seed_dir, "lung_gan_best_recall.pth"),
                    epoch + 1,
                    G,
                    D,
                    G_ema,
                    opt_G,
                    opt_D,
                    controller,
                    last_metrics,
                )

                print(f"  -> new best Recall ({rec_val:.4f})")

            if fid_val < best_fid:
                best_fid = fid_val

                best_fid_info = {
                    "epoch": epoch + 1,
                    "fid": fid_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "lpips_diversity": lpips_val,
                }

                save_checkpoint(
                    os.path.join(seed_dir, "lung_gan_best_fid.pth"),
                    epoch + 1,
                    G,
                    D,
                    G_ema,
                    opt_G,
                    opt_D,
                    controller,
                    last_metrics,
                )

                print(f"  -> new best FID ({fid_val:.2f})")

            if COMPUTE_PER_CLASS_METRICS:
                for c, cname in enumerate(dataset.classes):
                    n_avail = len(class_loaders[c].dataset)

                    f_c, p_c, r_c, l_c = compute_metrics(
                        G_ema.shadow,
                        class_loaders[c],
                        device,
                        fid_metric,
                        lpips_metric,
                        num_samples=min(METRICS_NUM_SAMPLES, n_avail),
                        z_dim=Z_DIM,
                        num_classes=NUM_CLASSES,
                        pr_k=PR_K,
                        lpips_num_pairs=LPIPS_NUM_PAIRS,
                        fixed_class=c,
                    )

                    metric_history["per_class"][cname]["epoch"].append(epoch + 1)
                    metric_history["per_class"][cname]["fid"].append(f_c)
                    metric_history["per_class"][cname]["precision"].append(p_c)
                    metric_history["per_class"][cname]["recall"].append(r_c)
                    metric_history["per_class"][cname]["lpips_diversity"].append(l_c)

                    print(
                        f"    [{cname:10s}] FID={f_c:.2f}  P={p_c:.4f}  "
                        f"R={r_c:.4f}  LPIPS={l_c:.4f}"
                    )

                recalls = {
                    cn: metric_history["per_class"][cn]["recall"][-1]
                    for cn in dataset.classes
                }

                leader = max(recalls, key=recalls.get)
                gap_leader = max(recalls.values()) - min(recalls.values())

                print(f"    Recall leader: {leader} (+{gap_leader:.4f})")

                per_class_recall_leader_history.append(
                    (epoch + 1, leader, gap_leader)
                )

        # Samples
        if SAVE_SAMPLES and ((epoch + 1) % SAMPLE_EVERY == 0 or epoch == 0):
            G_ema.shadow.eval()

            with torch.no_grad():
                fake = G_ema.shadow(fixed_z, fixed_labels).detach().cpu()

            grid = make_grid(fake, nrow=4, normalize=True, value_range=(-1, 1))

            plt.figure(figsize=(8, 8))
            plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
            plt.title(f"V15E Seed {seed} EMA samples — epoch {epoch + 1}")
            plt.axis("off")

            plt.savefig(
                os.path.join(seed_dir, f"samples_epoch_{epoch + 1}.png"),
                bbox_inches="tight",
            )
            plt.close()

            G_ema.shadow.train()

        # Periodic checkpoint
        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(
                os.path.join(seed_dir, f"lung_gan_epoch_{epoch + 1}.pth"),
                epoch + 1,
                G,
                D,
                G_ema,
                opt_G,
                opt_D,
                controller,
                last_metrics,
            )

    # Final checkpoint
    save_checkpoint(
        os.path.join(seed_dir, "lung_gan_final.pth"),
        EPOCHS,
        G,
        D,
        G_ema,
        opt_G,
        opt_D,
        controller,
        last_metrics,
    )

    # Compute budget summary for this seed
    w = CONTROLLER_WARMUP_EPOCHS

    if len(controller_history["n_dis"]) > w:
        post_nd = np.mean(controller_history["n_dis"][w:])
        post_msf = np.mean(controller_history["ms_frac"][w:])
    else:
        post_nd = np.mean(controller_history["n_dis"])
        post_msf = np.mean(controller_history["ms_frac"])

    d_savings = (1.0 - post_nd / 2.0) * 100.0
    ms_savings = (1.0 - post_msf) * 100.0

    if SAVE_HISTORY:
        torch.save(
            {
                "metric_history": metric_history,
                "loss_history": loss_history,
                "controller_history": controller_history,
                "per_class_recall_leader_history": per_class_recall_leader_history,
                "best_recall_info": best_recall_info,
                "best_fid_info": best_fid_info,
                "post_warmup_avg_n_dis": post_nd,
                "post_warmup_ms_frac": post_msf,
                "d_savings_pct": d_savings,
                "ms_savings_pct": ms_savings,
            },
            os.path.join(seed_dir, f"history_seed_{seed}.pt"),
        )

    total_time_min = (time.time() - start_time) / 60.0

    all_results.append({
        "seed": seed,
        "best_fid": best_fid,
        "best_recall": best_recall,
        "best_fid_precision": best_fid_info.get("precision", np.nan),
        "best_fid_recall": best_fid_info.get("recall", np.nan),
        "best_fid_lpips": best_fid_info.get("lpips_diversity", np.nan),
        "best_recall_fid": best_recall_info.get("fid", np.nan),
        "best_recall_precision": best_recall_info.get("precision", np.nan),
        "best_recall_lpips": best_recall_info.get("lpips_diversity", np.nan),
        "avg_n_dis": post_nd,
        "ms_frac": post_msf,
        "d_savings_pct": d_savings,
        "ms_savings_pct": ms_savings,
        "time_min": total_time_min,
    })

    print(f"Seed {seed} complete. Time: {total_time_min:.1f} min")
    print(f"Average N_DIS: {post_nd:.2f}")
    print(f"MS computed fraction: {post_msf:.1%}")
    print(f"D-step savings: {d_savings:.1f}%")
    print(f"MS savings: {ms_savings:.1f}%")

    del G, D, G_ema, opt_G, opt_D, fid_metric, lpips_metric
    del loader, class_loaders, controller

    gc.collect()
    torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
df = pd.DataFrame(all_results)

print("\n" + "=" * 80)
print("V15E FINAL MULTI-SEED SUMMARY")
print("=" * 80)
print(df.to_string(index=False))

print("\nMean ± Std:")
print(f"Best FID:             {df['best_fid'].mean():.2f} ± {df['best_fid'].std():.2f}")
print(f"Best Recall:          {df['best_recall'].mean():.4f} ± {df['best_recall'].std():.4f}")
print(f"Best-FID Precision:   {df['best_fid_precision'].mean():.4f} ± {df['best_fid_precision'].std():.4f}")
print(f"Best-FID LPIPS:       {df['best_fid_lpips'].mean():.4f} ± {df['best_fid_lpips'].std():.4f}")
print(f"Best-Recall FID:      {df['best_recall_fid'].mean():.2f} ± {df['best_recall_fid'].std():.2f}")
print(f"Best-Recall Precision:{df['best_recall_precision'].mean():.4f} ± {df['best_recall_precision'].std():.4f}")
print(f"Best-Recall LPIPS:    {df['best_recall_lpips'].mean():.4f} ± {df['best_recall_lpips'].std():.4f}")
print(f"Avg N_DIS:            {df['avg_n_dis'].mean():.2f} ± {df['avg_n_dis'].std():.2f}")
print(f"MS computed frac:     {df['ms_frac'].mean():.3f} ± {df['ms_frac'].std():.3f}")
print(f"D-step savings:       {df['d_savings_pct'].mean():.1f}% ± {df['d_savings_pct'].std():.1f}%")
print(f"MS savings:           {df['ms_savings_pct'].mean():.1f}% ± {df['ms_savings_pct'].std():.1f}%")
print(f"Time (min):           {df['time_min'].mean():.1f} ± {df['time_min'].std():.1f}")

df.to_csv(os.path.join(CKPT_ROOT, "v15E_multiseed_results.csv"), index=False)
print(f"\nSaved summary CSV to: {os.path.join(CKPT_ROOT, 'v15E_multiseed_results.csv')}")

Device: cuda
Drive already mounted.
Data root: /content/lung_data/Lung Segmentation Data V2/Train
Classes (2): ['COVID-19', 'Normal']
Dataset size: 14507

V15E | SEED 42
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


Seed 42 Epoch 1/200:   0%|          | 0/226 [00:00<?, ?it/s]

Seed 42 Epoch 1/200 | D=0.5040 | G=8.6549 | D(real)=4.499 D(fake)=-2.148 | ND=2.00 | MS=100.0% | effλ=0.110 | Gap=6.202
  Metrics: FID=524.95  P=0.0000  R=0.0000  LPIPS=0.0000
  -> new best Recall (0.0000)
  -> new best FID (524.95)
    [COVID-19  ] FID=534.83  P=0.0000  R=0.0000  LPIPS=0.0000
    [Normal    ] FID=514.49  P=0.0000  R=0.0000  LPIPS=0.0000
    Recall leader: COVID-19 (+0.0000)


Seed 42 Epoch 2/200:   0%|          | 0/226 [00:00<?, ?it/s]

Seed 42 Epoch 2/200 | D=1.4260 | G=0.4801 | D(real)=0.099 D(fake)=-0.650 | ND=2.00 | MS=100.0% | effλ=0.091 | Gap=1.096


...

Seed 777 Epoch 198/200 | D=1.8701 | G=0.2934 | D(real)=0.066 D(fake)=-0.077 | ND=1.00 | MS=46.9% | effλ=0.062 | Gap=0.143


Seed 777 Epoch 199/200:   0%|          | 0/226 [00:00<?, ?it/s]

Seed 777 Epoch 199/200 | D=1.8672 | G=0.3231 | D(real)=0.012 D(fake)=-0.133 | ND=1.00 | MS=40.3% | effλ=0.062 | Gap=0.143


Seed 777 Epoch 200/200:   0%|          | 0/226 [00:00<?, ?it/s]

Seed 777 Epoch 200/200 | D=1.8638 | G=0.2934 | D(real)=0.032 D(fake)=-0.114 | ND=1.00 | MS=38.9% | effλ=0.062 | Gap=0.146
  Metrics: FID=32.21  P=0.6810  R=0.2830  LPIPS=0.3343
  -> new best FID (32.21)
    [COVID-19  ] FID=36.99  P=0.6397  R=0.2521  LPIPS=0.3683
    [Normal    ] FID=31.18  P=0.6811  R=0.2453  LPIPS=0.3069
    Recall leader: COVID-19 (+0.0068)
Seed 777 complete. Time: 178.6 min
Average N_DIS: 1.11
MS computed fraction: 44.9%
D-step savings: 44.6%
MS savings: 55.1%

V15E FINAL MULTI-SEED SUMMARY
 seed  best_fid  best_recall  best_fid_precision  best_fid_recall  best_fid_lpips  best_recall_fid  best_recall_precision  best_recall_lpips  avg_n_dis  ms_frac  d_savings_pct  ms_savings_pct   time_min
   42 32.340199       0.3160              0.7220            0.273        0.334013        34.364788                 0.6495           0.333606   1.024819 0.450115      48.759040       54.988545 174.628820
  123 31.477047       0.2615              0.7495            0.219        0.33